In [10]:
import pandas as pd
from qiskit.quantum_info import SparsePauliOp, Statevector, Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis.evolution import LieTrotter, SuzukiTrotter
import numpy as np
from scipy.linalg import expm

In [11]:
csv_path = "ES_H4_linear_R1.2_sto-6g.csv"  
df = pd.read_csv(csv_path)
labels = df["label"].tolist()
coeffs = df["coef"].astype(float).tolist()
n_qubits = len(labels[0]) if labels else 0
print(f"Number of qubits: {n_qubits}")
print(f"Coefficients: {labels}")

Number of qubits: 8
Coefficients: ['IIIIIIII', 'IIIIYYXX', 'YZZZZYXX', 'IXZZXIXX', 'IIYYIIXX', 'YYIIIIXX', 'IIIIXYYX', 'XZZZZYYX', 'IXZZYIYX', 'IIXYIIYX', 'XYIIIIYX', 'IIXZXXZX', 'IIYZYXZX', 'IXZXIXZX', 'IYZYIXZX', 'XZXIIXZX', 'YZYIIXZX', 'IXZYIYZX', 'YZZYXZZX', 'IXXIXZZX', 'XZZYYZZX', 'IXYIYZZX', 'IIIXZZZX', 'IIZXZZZX', 'IZIXZZZX', 'ZIIXZZZX', 'YYXZZZZX', 'XYYZZZZX', 'IIIXIZZX', 'IIIXZIZX', 'IIIXZZIX', 'IIIIYXXY', 'YZZZZXXY', 'IYZZXIXY', 'IIYXIIXY', 'YXIIIIXY', 'IIIIXXYY', 'XZZZZXYY', 'IYZZYIYY', 'IIXXIIYY', 'XXIIIIYY', 'IYZXIXZY', 'IIXZXYZY', 'IIYZYYZY', 'IXZXIYZY', 'IYZYIYZY', 'XZXIIYZY', 'YZYIIYZY', 'YZZXXZZY', 'IYXIXZZY', 'XZZXYZZY', 'IYYIYZZY', 'IIIYZZZY', 'IIZYZZZY', 'IZIYZZZY', 'ZIIYZZZY', 'YXXZZZZY', 'XXYZZZZY', 'IIIYIZZY', 'IIIYZIZY', 'IIIYZZIY', 'IIIIIIIZ', 'IIXZZZXZ', 'IIYZZZYZ', 'IIIIIIZZ', 'IXZZZXIZ', 'IYZZZYIZ', 'IIIIIZIZ', 'XZZZXIIZ', 'YZZZYIIZ', 'IIIIZIIZ', 'IIIZIIIZ', 'IIZIIIIZ', 'IZIIIIIZ', 'ZIIIIIIZ', 'IIIYYXXI', 'XZZXIXXI', 'IYYIIXXI', 'IIIXYYXI', 'XZZYIYXI', 'IXYI

In [ ]:
H = SparsePauliOp.from_list(list(zip(labels, coeffs)))

T = 1.0
N = 100
rule = LieTrotter(reps=N)
U = PauliEvolutionGate(H, time=T, synthesis=rule)


hf_bitstring = "11110000"

# Build the statevector corresponding to that computational basis state:
sv0 = Statevector.from_label(hf_bitstring[::-1])  # Qiskit uses leftmost=highest qubit
svT = sv0.evolve(U)

Hmat = H.to_matrix()
U_exact = expm(-1j * T * Hmat)
sv_exact = Statevector(U_exact @ sv0.data)

# === 6) Compute Z-expectations and site occupations n_i = (1 - Z_up)/2 + (1 - Z_dn)/2 ===
# Here we assume spin-orbital indexing groups as sites: (i,↑)=2*i, (i,↓)=2*i+1  (adjust if needed)
def z_expect(sv, q):
    """<Z_q> with Pauli label where qubit 0 is rightmost char."""
    label = ['I'] * n_qubits
    label[q] = 'Z'
    return sv.expectation_value(Pauli(''.join(reversed(label)))).real

# Spin-orbital to site mapping (8 qubits -> 4 sites)
def site_occupations(sv):
    occs = []
    for i in range(n_qubits // 2):
        q_up = 2*i      # (i,↑)
        q_dn = 2*i + 1  # (i,↓)
        n_up = 0.5 * (1 - z_expect(sv, q_up))
        n_dn = 0.5 * (1 - z_expect(sv, q_dn))
        occs.append((i, float(n_up + n_dn)))
    return occs

print(f"#qubits: {n_qubits}, #terms: {len(labels)}")
print("Site occupations n_i(T):")
for i, ni in site_occupations(svT):
    print(f"  i={i}: n_i={ni:.6f}")
print("Site occupations n_i(T): Exact")
for i, ni in site_occupations(sv_exact):
    print(f"  i={i}: n_i={ni:.6f}")

#qubits: 8, #terms: 185
Site occupations n_i(T):
  i=0: n_i=1.930539
  i=1: n_i=1.912182
  i=2: n_i=0.089345
  i=3: n_i=0.067933
  i=0: n_i=1.930536
  i=1: n_i=1.912186
  i=2: n_i=0.089344
  i=3: n_i=0.067935
